In [ ]:
# Cell 1 — Kết nối Google Drive
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Cell 2 — Tự tìm đúng thư mục COCO_2014
from pathlib import Path

MYDRIVE = Path('/content/drive/MyDrive/Data')
json_candidates = sorted(
    p for p in MYDRIVE.rglob('dataset_coco.json')
    if p.is_file() and p.parent.name.lower() == 'karpathy'
)

root_candidates = []
for json_path in json_candidates:
    root = json_path.parent.parent
    if (root / 'train2014').is_dir() and (root / 'val2014').is_dir():
        root_candidates.append((root, json_path))

print('Các ứng viên tìm được:')
for i, (root, json_path) in enumerate(root_candidates):
    print(i, '| ROOT =', root, '| JSON =', json_path)

assert root_candidates, (
    'Không tìm thấy ROOT hợp lệ. Hãy kiểm tra quyền chia sẻ, shortcut và vị trí dataset_coco.json.'
)
assert len(root_candidates) == 1, (
    f'Có {len(root_candidates)} ứng viên; hãy chọn đúng ROOT thủ công.'
)

ROOT, KARPATHY_JSON = root_candidates[0]

TRAIN_DIR = ROOT / 'train2014'
VAL_DIR = ROOT / 'val2014'
TEST_DIR = ROOT / 'test2014'

# Lưu output vào My Drive của bạn để không phụ thuộc quyền ghi trên thư mục được chia sẻ.
OUT_DIR = MYDRIVE / 'Data' / 'knowledge_base_level1_karpathy'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('KARPATHY_JSON:', KARPATHY_JSON)
print('train2014:', TRAIN_DIR.is_dir())
print('val2014:', VAL_DIR.is_dir())
print('test2014:', TEST_DIR.is_dir())
print('output:', OUT_DIR)


Các ứng viên tìm được:
0 | ROOT = /content/drive/MyDrive/Data/Giai_nen_coco2014/COCO_2014 | JSON = /content/drive/MyDrive/Data/Giai_nen_coco2014/COCO_2014/karpathy/dataset_coco.json
ROOT: /content/drive/MyDrive/Data/Giai_nen_coco2014/COCO_2014
KARPATHY_JSON: /content/drive/MyDrive/Data/Giai_nen_coco2014/COCO_2014/karpathy/dataset_coco.json
train2014: True
val2014: True
test2014: True
output: /content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy


In [ ]:
# Cell 3 — Kiểm tra JSON và định nghĩa hàm làm sạch caption
import json
import re
import unicodedata
from pathlib import Path

assert ROOT.is_dir(), f'Không tìm thấy ROOT: {ROOT}'
assert KARPATHY_JSON.is_file(), f'Không tìm thấy JSON: {KARPATHY_JSON}'

with KARPATHY_JSON.open('r', encoding='utf-8') as f:
    dataset = json.load(f)

assert isinstance(dataset, dict) and 'images' in dataset
images = dataset['images']
print('Top-level keys:', list(dataset.keys()))
print('Number of image records:', len(images))
assert len(images) == 123287


# Quy tắc làm sạch deterministic, bảo toàn nội dung caption.
CLEANING_VERSION = 'nfkc_whitespace_v1'


def clean_caption(text):
    """Chuẩn hóa caption nhưng không tự ý xóa dấu câu hay đổi nội dung."""
    text = unicodedata.normalize('NFKC', '' if text is None else str(text))
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def image_candidates(item):
    """Tạo các đường dẫn có thể xảy ra, gồm dạng phẳng và dạng lồng."""
    filepath = str(item.get('filepath', '')).strip('/\\')
    filename = Path(str(item['filename'])).name
    base = ROOT / filepath
    candidates = [
        base / filename,
        base / filepath / filename,
    ]

    candidates.extend([
        ROOT / filename,
        ROOT / 'train2014' / 'train2014' / filename,
        ROOT / 'val2014' / 'val2014' / filename,
        ROOT / 'test2014' / 'test2014' / filename,
    ])

    seen = set()
    for path in candidates:
        path = path.resolve()
        if path not in seen:
            seen.add(path)
            yield path


def resolve_image_path(item):
    """Trả về đường dẫn thật; ưu tiên cấu trúc đúng filepath trong JSON."""
    for path in image_candidates(item):
        if path.is_file():
            return path
    return None


for item in images[:10]:
    resolved = resolve_image_path(item)
    print(item['filename'], '->', resolved)

print('Cleaning version:', CLEANING_VERSION)
print('Caption example:', clean_caption('  A\u00a0dog\\t on\\n a mat.  '))

Top-level keys: ['images', 'dataset']
Number of image records: 123287
COCO_val2014_000000391895.jpg -> /content/drive/.shortcut-targets-by-id/1BgzGlcKkhqoKvTOjL4puNehYxuKFgO60/Data/Giai_nen_coco2014/COCO_2014/val2014/COCO_val2014_000000391895.jpg
COCO_val2014_000000522418.jpg -> /content/drive/.shortcut-targets-by-id/1BgzGlcKkhqoKvTOjL4puNehYxuKFgO60/Data/Giai_nen_coco2014/COCO_2014/val2014/COCO_val2014_000000522418.jpg
COCO_val2014_000000184613.jpg -> /content/drive/.shortcut-targets-by-id/1BgzGlcKkhqoKvTOjL4puNehYxuKFgO60/Data/Giai_nen_coco2014/COCO_2014/val2014/COCO_val2014_000000184613.jpg
COCO_val2014_000000318219.jpg -> /content/drive/.shortcut-targets-by-id/1BgzGlcKkhqoKvTOjL4puNehYxuKFgO60/Data/Giai_nen_coco2014/COCO_2014/val2014/COCO_val2014_000000318219.jpg
COCO_val2014_000000554625.jpg -> /content/drive/.shortcut-targets-by-id/1BgzGlcKkhqoKvTOjL4puNehYxuKFgO60/Data/Giai_nen_coco2014/COCO_2014/val2014/COCO_val2014_000000554625.jpg
COCO_val2014_000000397133.jpg -> /content/dri

In [ ]:
# Cell 4 — Kiểm tra đúng Karpathy split và không trùng image ID
from collections import Counter
from itertools import combinations

expected = Counter({
    'train': 82783,
    'restval': 30504,
    'val': 5000,
    'test': 5000,
})

actual = Counter(item.get('split') for item in images)
print('Actual split counts:', dict(actual))
assert actual == expected, f'Sai số lượng split: {dict(actual)}'
assert len(images) == 123287

ids_by_split = {
    split: {item['cocoid'] for item in images if item.get('split') == split}
    for split in expected
}

for split, ids in ids_by_split.items():
    assert len(ids) == expected[split], f'Image ID trùng trong split {split}'

for left, right in combinations(expected, 2):
    overlap = ids_by_split[left] & ids_by_split[right]
    assert not overlap, f'Trùng image ID giữa {left} và {right}: {list(overlap)[:3]}'

print('PASS: đúng Karpathy split và không có image-ID overlap')
print('KB images train + restval:', expected['train'] + expected['restval'])


Actual split counts: {'test': 5000, 'restval': 30504, 'val': 5000, 'train': 82783}
PASS: đúng Karpathy split và không có image-ID overlap
KB images train + restval: 113287


In [ ]:
# Cell 5 — Kiểm tra ảnh, caption và kết quả làm sạch
from collections import Counter

image_count = Counter()
caption_count = Counter()
cleaning_changed_count = Counter()
empty_after_cleaning_count = Counter()
missing_count = 0
missing_examples = []

for item in images:
    split = item['split']
    image_count[split] += 1

    for sentence in item.get('sentences', []):
        raw = sentence.get('raw', '')
        cleaned = clean_caption(raw)

        if cleaned:
            caption_count[split] += 1
            if str(raw) != cleaned:
                cleaning_changed_count[split] += 1
        else:
            empty_after_cleaning_count[split] += 1

    if resolve_image_path(item) is None:
        missing_count += 1
        if len(missing_examples) < 20:
            missing_examples.append({
                'filename': item['filename'],
                'filepath': item.get('filepath'),
                'tried': [str(p) for p in image_candidates(item)],
            })

print('Image counts:', dict(image_count))
print('Caption counts after cleaning:', dict(caption_count))
print('Captions changed by cleaning:', dict(cleaning_changed_count))
print('Empty captions dropped after cleaning:', dict(empty_after_cleaning_count))
print('Cleaning version:', CLEANING_VERSION)
print('Missing count:', missing_count)
print('Missing examples shown:', len(missing_examples))
for row in missing_examples[:5]:
    print(row)

assert missing_count == 0, (
    'Có ảnh không tìm thấy; dừng lại để sửa đường dẫn/quyền truy cập.'
)
assert caption_count['train'] == 414113
assert caption_count['restval'] == 152634
assert caption_count['train'] + caption_count['restval'] == 566747

print('PASS: ảnh thật, caption sau làm sạch và số lượng mapping hợp lệ')

Image counts: {'test': 5000, 'restval': 30504, 'val': 5000, 'train': 82783}
Caption counts after cleaning: {'test': 25010, 'restval': 152634, 'val': 25010, 'train': 414113}
Captions changed by cleaning: {'test': 3783, 'restval': 23326, 'val': 3762, 'train': 63158}
Empty captions dropped after cleaning: {}
Cleaning version: nfkc_whitespace_v1
Missing count: 0
Missing examples shown: 0
PASS: ảnh thật, caption sau làm sạch và số lượng mapping hợp lệ


In [ ]:
# # Cell 6 — Tìm vị trí ảnh thiếu
# from pathlib import Path

# target = 'COCO_train2014_000000270248.jpg'
# matches = list(Path('/content/drive').rglob(target))

# print('Số vị trí tìm thấy:', len(matches))
# for p in matches:
#     print(p, '| is_file =', p.is_file())


In [ ]:
# # Cell 7 — Kiểm tra thực tế số hình ảnh
# import os
# from pathlib import Path

# # Không ghi đè ROOT đã được tự động xác định ở Cell 2.
# train_imgs = list((TRAIN_DIR / 'train2014').glob('*.jpg'))
# if not train_imgs:
#     train_imgs = list(TRAIN_DIR.glob('*.jpg'))

# val_imgs = list((VAL_DIR / 'val2014').glob('*.jpg'))
# if not val_imgs:
#     val_imgs = list(VAL_DIR.glob('*.jpg'))

# print(f'Số ảnh train2014 thực tế: {len(train_imgs)} / 82783')
# print(f'Số ảnh val2014 thực tế: {len(val_imgs)} / 40504')


Số ảnh train2014 thực tế: 82782 / 82783
Số ảnh val2014 thực tế: 40504 / 40504


In [ ]:
# Cell 8 — Tạo manifest chính thức với raw_caption và caption đã làm sạch
import json

manifest_dir = OUT_DIR / 'manifests'
manifest_dir.mkdir(parents=True, exist_ok=True)

all_manifest = manifest_dir / 'all_images.jsonl'
train_manifest = manifest_dir / 'train_restval_images.jsonl'
val_manifest = manifest_dir / 'val_images.jsonl'
test_manifest = manifest_dir / 'test_images.jsonl'
caption_manifest = manifest_dir / 'train_restval_captions.jsonl'

paths = {
    'all': all_manifest,
    'train_restval': train_manifest,
    'val': val_manifest,
    'test': test_manifest,
    'captions': caption_manifest,
}
handles = {name: path.open('w', encoding='utf-8') for name, path in paths.items()}

try:
    for item in images:
        resolved = resolve_image_path(item)
        assert resolved is not None, f"Không tìm thấy {item['filename']}"

        captions = []
        for sentence in item.get('sentences', []):
            raw_caption = sentence.get('raw', '')
            cleaned_caption = clean_caption(raw_caption)

            if cleaned_caption:
                captions.append({
                    'sentid': sentence.get('sentid'),
                    'raw_caption': str(raw_caption),
                    'text': cleaned_caption,
                    'cleaning_version': CLEANING_VERSION,
                })

        record = {
            'image_id': item['cocoid'],
            'imgid': item.get('imgid'),
            'filename': item['filename'],
            'filepath': item.get('filepath', ''),
            'image_path': str(resolved),
            'split': item['split'],
            'captions': captions,
        }
        line = json.dumps(record, ensure_ascii=False) + '\n'
        handles['all'].write(line)

        if item['split'] in {'train', 'restval'}:
            handles['train_restval'].write(line)

            for caption in captions:
                caption_record = {
                    'row_id': None,
                    'image_id': item['cocoid'],
                    'filename': item['filename'],
                    'filepath': item.get('filepath', ''),
                    'image_path': str(resolved),
                    'split': item['split'],
                    'caption_id': caption['sentid'],
                    'raw_caption': caption['raw_caption'],
                    'caption': caption['text'],
                    'cleaning_version': caption['cleaning_version'],
                }
                handles['captions'].write(
                    json.dumps(caption_record, ensure_ascii=False) + '\n'
                )

        elif item['split'] == 'val':
            handles['val'].write(line)
        elif item['split'] == 'test':
            handles['test'].write(line)

finally:
    for handle in handles.values():
        handle.close()

for name, path in paths.items():
    with path.open('r', encoding='utf-8') as f:
        count = sum(1 for _ in f)
    print(name, '|', path.name, '|', count, 'lines')

assert sum(1 for _ in caption_manifest.open('r', encoding='utf-8')) == 566747
print('PASS: manifest có raw_caption, caption sạch và cleaning_version tại', manifest_dir)

all | all_images.jsonl | 123287 lines
train_restval | train_restval_images.jsonl | 113287 lines
val | val_images.jsonl | 5000 lines
test | test_images.jsonl | 5000 lines
captions | train_restval_captions.jsonl | 566747 lines
PASS: manifest có raw_caption, caption sạch và cleaning_version tại /content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy/manifests


In [ ]:
# Cell 9 — Cài thư viện và nạp CLIP
!pip -q install transformers faiss-cpu pillow tqdm

import json
import time
from pathlib import Path

import numpy as np
import torch
from tqdm.auto import tqdm
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = 'openai/clip-vit-base-patch32'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 128 if DEVICE == 'cuda' else 32
CHUNK_SIZE = 10000

caption_manifest = OUT_DIR / 'manifests' / 'train_restval_captions.jsonl'
chunk_dir = OUT_DIR / 'embedding_chunks'
chunk_dir.mkdir(parents=True, exist_ok=True)
progress_path = OUT_DIR / 'embedding_progress.json'

with caption_manifest.open('r', encoding='utf-8') as f:
    caption_count_total = sum(1 for _ in f)

print('Device:', DEVICE)
print('Caption count:', caption_count_total)
assert caption_count_total == 566747

model = CLIPModel.from_pretrained(MODEL_NAME).to(DEVICE)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad = False


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 98.0 MB/s eta 0:00:00
Device: cuda
Caption count: 566747


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

... (3 outputs left out)

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

In [ ]:
# Cell 9.5 — Xóa artefact cũ để tạo lại embedding từ caption sạch
# import shutil

# old_artifacts = [
#     OUT_DIR / 'embedding_chunks',
#     OUT_DIR / 'text_embeddings_train_restval.npy',
#     OUT_DIR / 'text_embedding_mapping.jsonl',
#     OUT_DIR / 'text_embeddings_train_restval.faiss',
#     OUT_DIR / 'metadata.json',
#     OUT_DIR / 'retrieval_report.json',
#     OUT_DIR / 'embedding_progress.json',
# ]

# for path in old_artifacts:
#     if path.is_dir():
#         shutil.rmtree(path)
#         print('Removed directory:', path)
#     elif path.exists():
#         path.unlink()
#         print('Removed file:', path)

# print('PASS: đã dọn artefact cũ; Cell 10 sẽ tạo embedding từ caption sạch')

In [ ]:
# Cell 10 — Encode caption theo checkpoint, có thể resume
chunk_dir.mkdir(parents=True, exist_ok=True)
progress_path.parent.mkdir(parents=True, exist_ok=True)

def encode_texts(texts):
    """Mã hóa caption thành vector CLIP 512 chiều và L2-normalize."""
    outputs = []

    for batch_start in range(0, len(texts), BATCH_SIZE):
        batch = texts[batch_start:batch_start + BATCH_SIZE]

        inputs = processor(
            text=batch,
            return_tensors='pt',
            padding=True,
            truncation=True,
        )
        inputs = {
            key: value.to(DEVICE)
            for key, value in inputs.items()
        }

        with torch.inference_mode():
            # Không dùng model.get_text_features(**inputs)
            # vì một số phiên bản transformers trả về
            # BaseModelOutputWithPooling thay vì tensor.
            text_outputs = model.text_model(
                input_ids=inputs['input_ids'],
                attention_mask=inputs.get('attention_mask'),
            )

            # Lấy pooled output rồi chiếu vào không gian CLIP.
            features = model.text_projection(
                text_outputs.pooler_output
            )

            # Chuẩn hóa L2 để dùng với FAISS IndexFlatIP.
            features = features / features.norm(
                dim=-1,
                keepdim=True,
            )

        outputs.append(
            features.detach()
            .cpu()
            .numpy()
            .astype('float32')
        )

    return np.concatenate(outputs, axis=0)



def chunk_paths(start, end):
    return (
        chunk_dir / f'emb_{start:06d}_{end:06d}.npy',
        chunk_dir / f'meta_{start:06d}_{end:06d}.jsonl',
    )



def is_complete(emb_path, meta_path, expected_rows):
    """Chỉ coi chunk hoàn thành khi embedding và metadata đều hợp lệ."""
    if not emb_path.is_file() or not meta_path.is_file():
        return False

    try:
        arr = np.load(emb_path, mmap_mode='r')

        if arr.shape != (expected_rows, 512):
            return False
        if arr.dtype != np.float32:
            return False

        with meta_path.open('r', encoding='utf-8') as f:
            metadata_rows = sum(1 for _ in f)

        return metadata_rows == expected_rows

    except Exception:
        return False


completed = []
source_index = 0
rows = []
chunk_start = 0

with caption_manifest.open('r', encoding='utf-8') as source:
    for line in tqdm(source, total=caption_count_total):
        rows.append(json.loads(line))
        source_index += 1

        is_last = source_index == caption_count_total
        if len(rows) < CHUNK_SIZE and not is_last:
            continue

        chunk_end = source_index
        expected_rows = chunk_end - chunk_start
        emb_path, meta_path = chunk_paths(
            chunk_start,
            chunk_end,
        )

        if is_complete(
            emb_path,
            meta_path,
            expected_rows,
        ):
            print(
                f'SKIP chunk {chunk_start}:{chunk_end} '
                '— checkpoint hợp lệ đã tồn tại'
            )
            completed.append([chunk_start, chunk_end])

        else:
            texts = [row['caption'] for row in rows]
            features = encode_texts(texts)

            assert features.shape == (
                expected_rows,
                512,
            )
            assert features.dtype == np.float32
            assert np.isfinite(features).all()

            # Ghi file tạm trước; chỉ đổi tên sau khi ghi xong.
            # Nhờ vậy checkpoint bị ngắt giữa chừng sẽ không được xem
            # là chunk hoàn chỉnh.
            temp_emb = emb_path.with_suffix('.tmp.npy')
            temp_meta = meta_path.with_suffix('.tmp.jsonl')

            np.save(temp_emb, features)

            with temp_meta.open('w', encoding='utf-8') as meta_out:
                for offset, row in enumerate(rows):
                    row['row_id'] = chunk_start + offset
                    meta_out.write(
                        json.dumps(
                            row,
                            ensure_ascii=False,
                        ) + '\n'
                    )

            temp_emb.replace(emb_path)
            temp_meta.replace(meta_path)

            completed.append([chunk_start, chunk_end])
            print(
                f'SAVED checkpoint chunk '
                f'{chunk_start}:{chunk_end}'
            )

        progress = {
            'caption_total': caption_count_total,
            'chunk_size': CHUNK_SIZE,
            'batch_size': BATCH_SIZE,
            'model_name': MODEL_NAME,
            'embedding_dim': 512,
            'completed_chunks': completed,
            'updated_at': time.strftime(
                '%Y-%m-%d %H:%M:%S'
            ),
        }

        progress_path.write_text(
            json.dumps(
                progress,
                indent=2,
                ensure_ascii=False,
            ),
            encoding='utf-8',
        )

        rows = []
        chunk_start = chunk_end

print('Đã hoàn thành các checkpoint hợp lệ.')
print('Checkpoint directory:', chunk_dir)
print('Progress file:', progress_path)


  0%|          | 0/566747 [00:00<?, ?it/s]

SAVED checkpoint chunk 0:10000
SAVED checkpoint chunk 10000:20000
SAVED checkpoint chunk 20000:30000
SAVED checkpoint chunk 30000:40000
SAVED checkpoint chunk 40000:50000
SAVED checkpoint chunk 50000:60000
SAVED checkpoint chunk 60000:70000
SAVED checkpoint chunk 70000:80000
SAVED checkpoint chunk 80000:90000
SAVED checkpoint chunk 90000:100000
SAVED checkpoint chunk 100000:110000
SAVED checkpoint chunk 110000:120000
SAVED checkpoint chunk 120000:130000
SAVED checkpoint chunk 130000:140000
SAVED checkpoint chunk 140000:150000
SAVED checkpoint chunk 150000:160000
SAVED checkpoint chunk 160000:170000
SAVED checkpoint chunk 170000:180000
SAVED checkpoint chunk 180000:190000
SAVED checkpoint chunk 190000:200000
SAVED checkpoint chunk 200000:210000
SAVED checkpoint chunk 210000:220000
SAVED checkpoint chunk 220000:230000
SAVED checkpoint chunk 230000:240000
SAVED checkpoint chunk 240000:250000
SAVED checkpoint chunk 250000:260000
SAVED checkpoint chunk 260000:270000
SAVED checkpoint chunk 2

In [ ]:
# Cell 11 — Kiểm tra chunk liên tục và ghép embedding
import re

embedding_path = OUT_DIR / 'text_embeddings_train_restval.npy'
embedding_metadata_path = OUT_DIR / 'text_embedding_mapping.jsonl'

pattern = re.compile(r'^emb_(\d+)_(\d+)\.npy$')
chunk_specs = []
for path in chunk_dir.glob('emb_*.npy'):
    match = pattern.match(path.name)
    if match:
        chunk_specs.append((int(match.group(1)), int(match.group(2)), path))

chunk_specs.sort(key=lambda x: x[0])
assert chunk_specs, 'Chưa có embedding chunk nào.'

cursor = 0
for start, end, emb_path in chunk_specs:
    assert start == cursor, f'Thiếu hoặc lệch chunk tại {cursor}; gặp {start}:{end}'
    arr = np.load(emb_path, mmap_mode='r')
    assert arr.shape == (end - start, 512)
    assert arr.dtype == np.float32
    cursor = end
assert cursor == caption_count_total

meta_specs = []
for path in chunk_dir.glob('meta_*.jsonl'):
    match = re.match(r'^meta_(\d+)_(\d+)\.jsonl$', path.name)
    if match:
        meta_specs.append((int(match.group(1)), int(match.group(2)), path))
meta_specs.sort(key=lambda x: x[0])

assert [(a, b) for a, b, _ in meta_specs] == [
    (a, b) for a, b, _ in chunk_specs
]

merged = np.lib.format.open_memmap(
    embedding_path,
    mode='w+',
    dtype='float32',
    shape=(caption_count_total, 512),
)

cursor = 0
for start, end, emb_path in chunk_specs:
    arr = np.load(emb_path, mmap_mode='r')
    merged[cursor:end] = arr
    cursor = end
merged.flush()

with embedding_metadata_path.open('w', encoding='utf-8') as output:
    for start, end, meta_path in meta_specs:
        output.write(meta_path.read_text(encoding='utf-8'))

print('Embedding:', embedding_path)
print('Mapping:', embedding_metadata_path)
print('Shape:', merged.shape)
assert merged.shape == (566747, 512)


Embedding: /content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy/text_embeddings_train_restval.npy
Mapping: /content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy/text_embedding_mapping.jsonl
Shape: (566747, 512)


In [ ]:
# Cell 12 — Kiểm tra norm và xây FAISS
import faiss
import numpy as np
from tqdm.auto import tqdm

embeddings = np.load(embedding_path, mmap_mode='r')
assert embeddings.shape == (566747, 512)
assert embeddings.dtype == np.float32
assert np.isfinite(embeddings[:1000]).all()

sample_norms = np.linalg.norm(np.asarray(embeddings[:1000]), axis=1)
print('Norm min/max:', sample_norms.min(), sample_norms.max())
assert np.allclose(sample_norms, 1.0, atol=1e-3)

index_path = OUT_DIR / 'text_embeddings_train_restval.faiss'
index = faiss.IndexFlatIP(512)
add_chunk_size = 8192
for start in tqdm(range(0, len(embeddings), add_chunk_size)):
    end = min(start + add_chunk_size, len(embeddings))
    index.add(np.asarray(embeddings[start:end], dtype='float32'))

faiss.write_index(index, str(index_path))
print('Saved index:', index_path)
print('ntotal:', index.ntotal)
print('d:', index.d)
assert index.ntotal == 566747
assert index.d == 512


Norm min/max: 0.9999999 1.0000001


  0%|          | 0/70 [00:00<?, ?it/s]

Saved index: /content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy/text_embeddings_train_restval.faiss
ntotal: 566747
d: 512


In [ ]:
# Cell 13 — Reload index, mapping và kiểm tra alignment
import json
import faiss
import numpy as np

reloaded_index = faiss.read_index(str(index_path))
with embedding_metadata_path.open('r', encoding='utf-8') as f:
    mapping_rows = [json.loads(line) for line in f]

mapping_count = len(mapping_rows)
print('index.ntotal:', reloaded_index.ntotal)
print('mapping count:', mapping_count)
print('embedding rows:', embeddings.shape[0])
assert reloaded_index.ntotal == 566747
assert mapping_count == 566747
assert embeddings.shape[0] == 566747
assert all(row['row_id'] == i for i, row in enumerate(mapping_rows))
print('PASS: index, embedding và mapping cùng số dòng')

# Kiểm tra nội dung alignment trên 5 vị trí cố định, không thay đổi artefact.
alignment_indices = np.linspace(0, mapping_count - 1, num=5, dtype=int).tolist()
alignment_texts = [mapping_rows[i]['caption'] for i in alignment_indices]
alignment_vectors = encode_texts(alignment_texts)
stored_vectors = np.asarray(embeddings[alignment_indices], dtype='float32')
alignment_cosines = np.sum(alignment_vectors * stored_vectors, axis=1)
print('Alignment cosine min/max:', alignment_cosines.min(), alignment_cosines.max())
assert np.allclose(alignment_cosines, 1.0, atol=1e-4)
print('PASS: caption trong mapping khớp embedding cùng row_id')


index.ntotal: 566747
mapping count: 566747
embedding rows: 566747
PASS: index, embedding và mapping cùng số dòng
Alignment cosine min/max: 1.0 1.0
PASS: caption trong mapping khớp embedding cùng row_id


In [ ]:
# Cell 14 — Ghi metadata cho gói bàn giao
import faiss
import hashlib
import json
import platform
import sys
from datetime import datetime, timezone
import transformers

metadata_path = OUT_DIR / 'metadata.json'
dataset_json_sha256 = hashlib.sha256(KARPATHY_JSON.read_bytes()).hexdigest()
created_at_utc = datetime.now(timezone.utc).isoformat()
metadata = {
    'dataset': 'MS COCO 2014',
    'split_source': 'Karpathy dataset_coco.json',
    'included_splits': ['train', 'restval'],
    'excluded_splits': ['val', 'test'],
    'image_count': 113287,
    'caption_count': 566747,
    'model_name': MODEL_NAME,
    'clip_model_revision': getattr(model.config, '_commit_hash', None),
    'clip_processor_name_or_path': getattr(processor, 'name_or_path', MODEL_NAME),
    'embedding_dim': 512,
    'dtype': 'float32',
    'normalization': 'L2',
    'caption_cleaning': {
    'version': CLEANING_VERSION,
    'unicode_normalization': 'NFKC',
    'whitespace': 'collapse consecutive whitespace and strip outer whitespace',
    'preserve_punctuation': True,
    'preserve_raw_caption': True,
    'drop_empty_after_cleaning': True,
    },
    'index_type': 'faiss.IndexFlatIP',
    'nested_directory_resolver': True,
    'root_used': str(ROOT),
    'dataset_json_sha256': dataset_json_sha256,
    'python_version': sys.version,
    'platform_python_version': platform.python_version(),
    'transformers_version': transformers.__version__,
    'pytorch_version': torch.__version__,
    'faiss_version': getattr(faiss, '__version__', 'unknown'),
    'chunk_size': CHUNK_SIZE,
    'batch_size': BATCH_SIZE,
    'mapping_order': 'Thứ tự dòng trong manifests/train_restval_captions.jsonl sau khi encode.',
    'retrieval_policy': {
        'default_overfetch': 50,
        'same_image_filter': 'Loại image_id nằm trong exclude_image_ids trước khi giữ đủ top_k.',
        'source_image_deduplication': 'Tùy chọn qua deduplicate_image_ids; mặc định giữ nguyên từng caption.',
    },
    'created_at_utc': created_at_utc,
}
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False),
    encoding='utf-8',
)
print(metadata_path)


/content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy/metadata.json


In [ ]:
# Cell 15 — Retrieval thử và kiểm tra leakage

import json
import numpy as np
import torch
from PIL import Image
from tqdm.auto import tqdm


def encode_image(image):
    """Mã hóa ảnh thành vector CLIP 512 chiều và L2-normalize."""
    image_inputs = processor(
        images=image,
        return_tensors='pt',
    )
    image_inputs = {
        key: value.to(DEVICE)
        for key, value in image_inputs.items()
    }

    with torch.inference_mode():
        # Lấy đầu ra vision model dạng tensor.
        vision_outputs = model.vision_model(
            pixel_values=image_inputs['pixel_values']
        )

        # Chiếu đặc trưng ảnh vào không gian CLIP 512 chiều.
        query_vector = model.visual_projection(
            vision_outputs.pooler_output
        )

        # Chuẩn hóa giống như text embedding ở Cell 10.
        query_vector = query_vector / query_vector.norm(
            dim=-1,
            keepdim=True,
        )

    return query_vector.detach().cpu().numpy().astype('float32')


def retrieve_from_vector(
    query_vector,
    top_k=5,
    exclude_image_ids=None,
    overfetch=50,
    deduplicate_image_ids=False,
):
    """Trả về caption, image_id, split, score và row_id từ một query vector."""
    query_vector = np.asarray(query_vector, dtype='float32')
    if query_vector.ndim == 1:
        query_vector = query_vector[None, :]
    assert query_vector.shape == (1, 512), query_vector.shape

    exclude_image_ids = {int(x) for x in (exclude_image_ids or set())}
    search_k = min(max(top_k, overfetch), reloaded_index.ntotal)
    scores, row_ids = reloaded_index.search(query_vector, search_k)
    results = []
    seen_image_ids = set()

    for score, row_id in zip(scores[0], row_ids[0]):
        row_id = int(row_id)
        if row_id < 0:
            continue
        row = mapping_rows[row_id]
        image_id = int(row['image_id'])
        if image_id in exclude_image_ids:
            continue
        if deduplicate_image_ids and image_id in seen_image_ids:
            continue

        result = {
            'row_id': row_id,
            'score': float(score),
            'image_id': image_id,
            'source_split': row['split'],
            'caption': row['caption'],
        }
        results.append(result)
        seen_image_ids.add(image_id)
        if len(results) == top_k:
            break

    return results


def retrieve(
    image,
    top_k=5,
    exclude_image_ids=None,
    overfetch=50,
    deduplicate_image_ids=False,
):
    """Encode ảnh rồi truy hồi caption từ knowledge base."""
    if isinstance(image, (str, Path)):
        with Image.open(image) as opened:
            image = opened.convert('RGB')
    query_vector = encode_image(image)
    return retrieve_from_vector(
        query_vector,
        top_k=top_k,
        exclude_image_ids=exclude_image_ids,
        overfetch=overfetch,
        deduplicate_image_ids=deduplicate_image_ids,
    )


# Chọn một ảnh thuộc split val làm query.
# Ảnh val không được nằm trong FAISS index.
query_item = next(
    item for item in images
    if item['split'] == 'val'
)

query_path = resolve_image_path(query_item)
assert query_path is not None, (
    f"Không tìm thấy ảnh query: {query_item['filename']}"
)

query_image = Image.open(query_path).convert('RGB')
query_vector = encode_image(query_image)

assert query_vector.shape == (1, 512)
assert query_vector.dtype == np.float32
assert np.isfinite(query_vector).all()
assert np.allclose(
    np.linalg.norm(query_vector, axis=1),
    1.0,
    atol=1e-3,
)

# Overfetch 50 rồi lọc same-image; query val cũng truyền policy này một cách tường minh.
retrieved_val = retrieve(
    query_image,
    top_k=5,
    exclude_image_ids={query_item['cocoid']},
    overfetch=50,
)
assert len(retrieved_val) == 5

print('Query image:', query_item['filename'])
print('Query image path:', query_path)
print('Query split:', query_item['split'])
print()

results = []
for rank, row in enumerate(retrieved_val, start=1):
    assert row['source_split'] in {'train', 'restval'}, (
        f"Nguồn không hợp lệ: {row['source_split']}"
    )
    assert row['image_id'] != query_item['cocoid'], (
        'Phát hiện self-retrieval cùng image_id.'
    )

    result = {'rank': rank, **row}
    results.append(result)

    print(
        f"Rank {rank} | "
        f"score={row['score']:.4f} | "
        f"image_id={row['image_id']} | "
        f"split={row['source_split']}"
    )
    print(' ', row['caption'])

# Kiểm tra trường hợp quan trọng: query thuộc train và phải loại cùng image_id.
train_query_item = next(
    item for item in images
    if item['split'] == 'train'
)
train_query_path = resolve_image_path(train_query_item)
assert train_query_path is not None
train_results = retrieve(
    train_query_path,
    top_k=5,
    exclude_image_ids={train_query_item['cocoid']},
    overfetch=50,
)
assert len(train_results) == 5
for row in train_results:
    assert row['source_split'] in {'train', 'restval'}
    assert row['image_id'] != train_query_item['cocoid']
print('PASS: train query đã loại caption cùng image_id với query')

# Kiểm thử retrieval trên 100 query val để kiểm tra contract nhiều lần.
multi_query_items = [
    item for item in images
    if item['split'] == 'val'
][:100]
multi_query_checked = 0
for item in tqdm(multi_query_items, desc='Kiểm thử 100 query val'):
    path = resolve_image_path(item)
    assert path is not None
    rows = retrieve(
        path,
        top_k=5,
        exclude_image_ids={item['cocoid']},
        overfetch=50,
    )
    assert len(rows) == 5
    assert all(row['source_split'] in {'train', 'restval'} for row in rows)
    assert all(row['image_id'] != item['cocoid'] for row in rows)
    multi_query_checked += 1
print('PASS: retrieval nhiều query:', multi_query_checked)

# Lưu báo cáo retrieval để bàn giao cho nhóm.
retrieval_report = {
    'query_image_id': query_item['cocoid'],
    'query_filename': query_item['filename'],
    'query_path': str(query_path),
    'query_split': query_item['split'],
    'top_k': 5,
    'overfetch': 50,
    'results': results,
    'leakage_check': 'PASS',
    'train_query_check': {
        'query_image_id': train_query_item['cocoid'],
        'query_filename': train_query_item['filename'],
        'query_split': train_query_item['split'],
        'same_image_excluded': True,
        'result_count': len(train_results),
    },
    'multi_query_check': {
        'query_split': 'val',
        'query_count': multi_query_checked,
        'same_image_excluded': True,
        'status': 'PASS',
    },
}

retrieval_report_path = OUT_DIR / 'retrieval_report.json'
retrieval_report_path.write_text(
    json.dumps(
        retrieval_report,
        indent=2,
        ensure_ascii=False,
    ),
    encoding='utf-8',
)

print()
print('Retrieval report:', retrieval_report_path)
print(
    'PASS: retrieval có indexed mapping, overfetch, same-image filter '
    'và kiểm thử nhiều query'
)


Query image: COCO_val2014_000000184613.jpg
Query image path: /content/drive/.shortcut-targets-by-id/1BgzGlcKkhqoKvTOjL4puNehYxuKFgO60/Data/Giai_nen_coco2014/COCO_2014/val2014/COCO_val2014_000000184613.jpg
Query split: val

Rank 1 | score=0.3460 | image_id=121731 | split=train
  A man holding a black umbrella herding cattle.
Rank 2 | score=0.3376 | image_id=354242 | split=restval
  a young boy holding onto a harness for a cow
Rank 3 | score=0.3371 | image_id=31471 | split=restval
  A woman tending to a brown cow with a rope wrapped around it's mouth.
Rank 4 | score=0.3346 | image_id=121731 | split=train
  a man leading a cow on a field holding an umbrella.
Rank 5 | score=0.3337 | image_id=2963 | split=train
  Man standing in a field photographs animal-shaped kites.
PASS: train query đã loại caption cùng image_id với query


Kiểm thử 100 query val:   0%|          | 0/100 [00:00<?, ?it/s]

PASS: retrieval nhiều query: 100

Retrieval report: /content/drive/MyDrive/Data/Data/knowledge_base_level1_karpathy/retrieval_report.json
PASS: retrieval có indexed mapping, overfetch, same-image filter và kiểm thử nhiều query
